# Dasar Machine Learning

## Bagian 1a: Membaca Data


### Tujuan Pembelajaran

- Membuat koneksi ke sebuah basis data SQL contoh, lalu membaca isinya.
- Mengenal parameter umum yang dipakai saat membaca data dari basis data.

### Pustaka

- `pandas`
- `sqlite3`


## Membaca data secara sederhana

Structured Query Language (SQL) adalah standar ANSI yang dipakai banyak basis data. SQL memudahkan kita mengambil data dari basis data besar dengan cara yang konsisten, meskipun setiap produk punya sedikit perbedaan. Pada latihan ini kita memakai SQLite, versi SQL yang ringan dan cukup terbatas, sehingga cocok untuk contoh sederhana.


In [ ]:
# Imports
import sqlite3
from itertools import islice

import pandas as pd
from IPython.display import display


### Koneksi ke basis data

Langkah pertama adalah membuat koneksi ke basis data SQL. Beberapa basis data yang umum dipakai bersama Python antara lain Microsoft SQL Server, PostgreSQL, MySQL, AWS Redshift, AWS Aurora, Oracle DB, Teradata, dan Db2. Masing-masing membutuhkan konfigurasi serta kredensial yang berbeda.

Di sini kita memakai `sqlite3` karena tidak memerlukan server maupun kata sandi. Paket koneksi lain yang sering dipakai adalah `SQLAlchemy`, `psycopg2`, dan `MySQLdb`.


In [ ]:
# Buka koneksi ke basis data lokal
db_connection = sqlite3.connect("data/classic_rock.db")
db_connection

### Membaca isi tabel

Setelah koneksi terbuka, kita bisa menjalankan query dan memuat hasilnya sebagai DataFrame pandas.


In [ ]:
# Ambil seluruh baris dari tabel rock_songs
select_all = """
SELECT *
FROM rock_songs;
"""

song_records = pd.read_sql(select_all, db_connection)
song_records.head()


In [ ]:
# Ringkas jumlah lagu dan rata-rata jumlah pemutaran per artis dan tahun rilis
summary_query = """
SELECT Artist, Release_Year, COUNT(*) AS num_songs, AVG(PlayCount) AS avg_plays
FROM rock_songs
GROUP BY Artist, Release_Year
ORDER BY num_songs DESC;
"""

artist_metrics = pd.read_sql(summary_query, db_connection)
artist_metrics.head()


## Parameter umum

Beberapa parameter yang sering dipakai saat membaca data SQL:

- `coerce_float`: memaksa nilai numerik menjadi float bila memungkinkan.
- `parse_dates`: daftar kolom yang diubah menjadi tipe tanggal.
- `chunksize`: jumlah baris pada setiap potongan hasil query.

Kolom `Release_Year` hanya berisi tahun, jadi contoh berikut membentuk kolom tanggal penuh lewat `printf` di SQL. Dengan begitu `parse_dates` menerima teks berformat `YYYY-MM-DD`, bukan angka tahun yang bisa salah tafsir.


In [ ]:
# Baca ringkasan per potongan lima baris
chunked_query = """
SELECT
    Artist,
    printf('%04d-01-01', Release_Year) AS release_date,
    COUNT(*) AS num_songs,
    AVG(PlayCount) AS avg_plays
FROM rock_songs
GROUP BY Artist, Release_Year
ORDER BY num_songs DESC;
"""

song_chunks = pd.read_sql(
    chunked_query,
    db_connection,
    coerce_float=True,
    parse_dates=["release_date"],
    chunksize=5,
)

chunk_list = list(islice(song_chunks, 5))
print(f"Menampilkan {len(chunk_list)} potongan")
display(*chunk_list)


## Ringkasan

`pandas.read_sql` memungkinkan hasil query SQL langsung dibaca sebagai DataFrame. Parameter `coerce_float`, `parse_dates`, dan `chunksize` membantu mengatur tipe data serta ukuran hasil, terutama saat tabel yang dibaca berukuran besar.
